# Balanda (Njo Viri) → English Fine-tuning

Fine-tunes **facebook/nllb-200-distilled-600M** on the Balanda language dataset  
using HuggingFace Transformers + PEFT/LoRA, evaluates with **ChrF**, and pushes  
adapter weights to HuggingFace Hub.

**Dataset split**: train=3669 | val=431 | test=217  
**Runtime**: Kaggle T4 GPU (≈ 2–3 h for 5 epochs)


## 1. Install dependencies

In [ ]:
%%capture
!pip install -q transformers==4.41.2 datasets peft accelerate bitsandbytes \
               sentencepiece sacrebleu huggingface_hub evaluate

## 2. Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # ── Model ──────────────────────────────────────────────────────────────
    model_name: str       = "facebook/nllb-200-distilled-600M"
    # Alternatively: "Helsinki-NLP/opus-mt-en-mul"

    # NLLB language codes (ignored if using opus-mt)
    src_lang: str         = "eng_Latn"   # English
    tgt_lang: str         = "zul_Latn"   # closest supported code; outputs Balanda
    # NOTE: Balanda has no NLLB code – we use Zulu as the nearest Bantu seed
    # and fine-tune the adapter to produce Balanda text.

    # ── Data ───────────────────────────────────────────────────────────────
    data_dir: str         = "/kaggle/input/balanda-dataset"
    # ^ upload train/val/test.jsonl as a Kaggle Dataset named 'balanda-dataset'

    # ── Training ───────────────────────────────────────────────────────────
    output_dir: str       = "/kaggle/working/nllb-balanda-lora"
    num_epochs: int       = 5
    batch_size: int       = 8
    grad_accum: int       = 4          # effective batch = 32
    lr: float             = 3e-4
    max_src_len: int      = 128
    max_tgt_len: int      = 128
    warmup_steps: int     = 100
    fp16: bool            = True
    seed: int             = 42

    # ── LoRA ───────────────────────────────────────────────────────────────
    lora_r: int           = 16
    lora_alpha: int       = 32
    lora_dropout: float   = 0.05
    # Target modules for NLLB encoder-decoder
    lora_target_modules: tuple = ("q_proj", "v_proj")

    # ── HuggingFace Hub ────────────────────────────────────────────────────
    hf_repo_id: str       = "YOUR_HF_USERNAME/nllb-balanda-lora"
    push_to_hub: bool     = True

cfg = Config()
print(cfg)

## 3. HuggingFace Hub login

In [ ]:
from huggingface_hub import login
import os

# Add your HF token as a Kaggle Secret named HF_TOKEN
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
else:
    print("⚠  No HF_TOKEN found – Hub push will be skipped.")
    cfg.push_to_hub = False

## 4. Load & preprocess dataset

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl(os.path.join(cfg.data_dir, "train.jsonl"))
val_data   = load_jsonl(os.path.join(cfg.data_dir, "val.jsonl"))
test_data  = load_jsonl(os.path.join(cfg.data_dir, "test.jsonl"))

print(f"train={len(train_data)}  val={len(val_data)}  test={len(test_data)}")
print("Sample:", train_data[0])

In [ ]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

# For NLLB we must set the forced_bos_token to the target language
is_nllb = "nllb" in cfg.model_name.lower()

def preprocess(examples):
    """
    For each record:
      - source = English (the 'output' field, since most pairs are Balanda→EN)
      - target = Balanda (the 'input' field)
    We train EN→Balanda so the model can translate into Balanda.
    Flip src/tgt here if you want Balanda→EN inference.
    """
    src_texts = [rec["output"] for rec in examples]  # English
    tgt_texts = [rec["input"]  for rec in examples]  # Balanda

    if is_nllb:
        tokenizer.src_lang = cfg.src_lang

    model_inputs = tokenizer(
        src_texts,
        max_length=cfg.max_src_len,
        truncation=True,
        padding="max_length",
    )

    if is_nllb:
        tokenizer.src_lang = cfg.tgt_lang

    with tokenizer.as_target_tokenizer() if not is_nllb else __import__("contextlib").nullcontext():
        labels = tokenizer(
            tgt_texts,
            max_length=cfg.max_tgt_len,
            truncation=True,
            padding="max_length",
        )

    label_ids = [
        [(-100 if t == tokenizer.pad_token_id else t) for t in ids]
        for ids in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

def make_dataset(records):
    ds = Dataset.from_list(records)
    # batch preprocess
    ds = ds.map(
        lambda batch: preprocess(batch),
        batched=True,
        batch_size=256,
        remove_columns=ds.column_names,
    )
    ds.set_format("torch")
    return ds

train_ds = make_dataset(train_data)
val_ds   = make_dataset(val_data)
test_ds  = make_dataset(test_data)
print("Tokenisation done.")

## 5. Load model with LoRA

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float16 if cfg.fp16 else torch.float32,
    device_map="auto",
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.lora_target_modules),
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 6. ChrF evaluation metric

In [ ]:
import evaluate
import numpy as np

chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Decode predictions
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Decode labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]  # list of reference lists

    result = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        word_order=2,   # ChrF++ (word n-grams up to 2)
    )
    return {"chrf": round(result["score"], 4)}

## 7. Training

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    warmup_steps=cfg.warmup_steps,
    fp16=cfg.fp16,
    predict_with_generate=True,
    generation_max_length=cfg.max_tgt_len,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
    seed=cfg.seed,
    push_to_hub=False,   # we push manually after training
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 8. Evaluate on test set

In [ ]:
test_results = trainer.evaluate(test_ds, metric_key_prefix="test")
print("Test results:", test_results)

## 9. Inference demo

In [ ]:
def translate_en_to_balanda(english_text: str) -> str:
    """Translate English → Balanda using the fine-tuned adapter."""
    if is_nllb:
        tokenizer.src_lang = cfg.src_lang
    inputs = tokenizer(english_text, return_tensors="pt").to(model.device)

    forced_bos = (
        tokenizer.lang_code_to_id[cfg.tgt_lang]
        if is_nllb
        else None
    )
    gen_kwargs = dict(max_new_tokens=128, num_beams=4)
    if forced_bos is not None:
        gen_kwargs["forced_bos_token_id"] = forced_bos

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

examples = [
    "I will eat fish",
    "She is drinking water",
    "We did not go to the market",
]
for en in examples:
    print(f"EN : {en}")
    print(f"BAL: {translate_en_to_balanda(en)}")
    print()

## 10. Save adapter & push to HuggingFace Hub

In [ ]:
# Save adapter weights locally
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"Adapter saved to {cfg.output_dir}")

if cfg.push_to_hub:
    from huggingface_hub import HfApi
    api = HfApi()

    # Create repo if needed
    api.create_repo(cfg.hf_repo_id, exist_ok=True, private=False)

    # Upload adapter
    model.push_to_hub(cfg.hf_repo_id)
    tokenizer.push_to_hub(cfg.hf_repo_id)
    print(f"Pushed to https://huggingface.co/{cfg.hf_repo_id}")
else:
    print("Hub push skipped (no token or push_to_hub=False).")

## 11. Load adapter later (inference-only)

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
model = PeftModel.from_pretrained(base, "YOUR_HF_USERNAME/nllb-balanda-lora")
tokenizer = AutoTokenizer.from_pretrained("YOUR_HF_USERNAME/nllb-balanda-lora")
```
